# 05 — Application: the proposed JetBlue–Spirit merger

JetBlue (B6) agreed to acquire Spirit (NK) in July 2022; a federal court blocked the deal in January 2024 and the
agreement was terminated in March 2024. We simulate the merger on the latest data (2025Q2) with the demand models
estimated on the full sample (notebooks 03 and 03b): marginal costs from the pre-merger first-order conditions,
B6 and NK products priced jointly afterwards, rivals re-optimising.

Outputs: predicted fare changes by route, consumer-surplus losses, the routes with the largest harm, and the
marginal-cost reductions (efficiencies) that would offset the price effects. The four latest quarters are also
simulated to check that results do not hinge on the season.

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
PARTIES = ("B6", "NK")
IV = IV_SETS["baseline"]
SEASONS = QUARTERS[-4:]
LATEST = qlabel(*LAST_Q)
p_all = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
est = load_json("demand_full_sample")
het = load_json("demand_heterogeneity") if (RES / f"demand_heterogeneity{'_smoke' if SMOKE else ''}.json").exists() else None
sample = p_all[p_all.quarter.isin({qlabel(*q) for q in SEASONS})].sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
brackets = pd.read_parquet(CLEAN / "acs_income_brackets.parquet")
INCOME = (brackets, [0, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 75, 100, 125, 150, 200, 400])
L = est["linear"]
MODELS = {"Logit": ("logit", {"alpha": L["logit"]["alpha"]}, {}),
          "NL (air travel)": ("nested_air", {"alpha": L["nested_air"]["alpha"], "rho": L["nested_air"]["rho"]}, {}),
          "RC demand+supply": ("rc", {"sigma": est["rc_supply"]["sigma"], "pi": est["rc_supply"]["pi"]}, {})}
if het is not None:
    MODELS["RC + income"] = ("rc_income", {"sigma": het["rc_income"]["sigma"], "pi": het["rc_income"]["pi"]}, {"income": INCOME})
    MODELS["Business/leisure types"] = ("rc_types", {"sigma": [0, 0, 0], "pi": het["rc_types"]["pi"]},
                                         {"business_share": het["rc_types"]["business_share"]})
PREF = "RC demand+supply"

def overlap_markets(d, quarter):
    both = d[d.quarter == quarter].groupby("market_ids").carrier.agg(lambda s: set(PARTIES) <= set(s))
    return both.index[both].tolist()

ov_all = [m for q in SEASONS for m in overlap_markets(sample, qlabel(*q))]
sample = sample[sample.market_ids.isin(ov_all)].reset_index(drop=True)     # only overlap markets can change
results = {}
for name, (kind, prm, kw) in MODELS.items():
    with pyblp.parallel(N_CORES):
        results[name] = evaluate_fixed(sample, kind, prm, IV, **kw)
ov_latest = overlap_markets(sample, LATEST)
ovp = sample[sample.market_ids.isin(ov_latest)]
ns_overlap = ovp.groupby("market_ids").apply(lambda d: d[d.carrier.isin(PARTIES) & (d.nonstop == 1)].carrier.nunique() == 2)
print(f"{LATEST}: {len(ov_latest)} overlap markets; {int(ns_overlap.sum())} with nonstop service by both carriers; "
      f"B6+NK passengers {ovp[ovp.carrier.isin(PARTIES)].passengers.sum()/1e6:.2f}m per quarter")

## Simulation in the latest quarter

In [ ]:
route_res, prod_res = {}, {}
for name, r in results.items():
    sim, cs = simulate_merger(r, sample, PARTIES, markets=ov_latest)
    rs = route_summary(sim, cs, PARTIES)
    rs["nonstop_overlap"] = rs.market_ids.map(ns_overlap).astype(bool)
    route_res[name], prod_res[name] = rs, sim

def summarize(rs, sim):
    pr = sim[sim.carrier.isin(PARTIES)]
    wa = lambda c: np.average(pr[pr.carrier == c].dprice_pct, weights=pr[pr.carrier == c].passengers)
    nso = rs[rs.nonstop_overlap]
    return {"overlap routes": len(rs),
            "B6/NK fare change, mean (%)": party_avg(rs),
            "B6 fare change (%)": wa("B6"), "NK fare change (%)": wa("NK"),
            "rival fare change (%)": np.average(rs.dprice_rivals.fillna(0), weights=rs.passengers * (1 - rs.party_share)),
            "nonstop-overlap routes: B6/NK change (%)": party_avg(nso) if len(nso) else np.nan,
            "routes with B6/NK increase > 5%": int((rs.dprice_parties > 5).sum()),
            "consumer surplus change ($m per quarter)": rs.dcs_usd.sum() / 1e6,
            "consumer surplus change ($m per year)": 4 * rs.dcs_usd.sum() / 1e6}

summary = pd.DataFrame({k: summarize(route_res[k], prod_res[k]) for k in results})
save_table(summary, "b6nk_summary", float_format="%.2f",
           note=f"Merger simulation in {LATEST}: Bertrand--Nash pricing, constant marginal costs. Fare changes passenger-weighted; "
                "annual consumer-surplus change is four times the quarterly change.")
summary

In [ ]:
rs = route_res[PREF].copy()
for name in results:
    rs = rs.merge(route_res[name][["market_ids", "dprice_parties"]].rename(columns={"dprice_parties": f"dp {name}"}), on="market_ids")
rs.to_csv(RES / "b6nk_route_predictions.csv", index=False)
harm = rs.sort_values("dcs_usd").head(20)
harm_tab = harm.assign(**{"CS loss ($m/yr)": -4 * harm.dcs_usd / 1e6, "pax (000/qtr)": harm.passengers / 1e3,
                          "B6/NK share": harm.party_share, "B6/NK fare ($)": harm.fare_parties,
                          "both nonstop": harm.nonstop_overlap.astype(int)}).set_index("route")
harm_tab = harm_tab[["B6/NK fare ($)", "B6/NK share", "both nonstop", "pax (000/qtr)"] + [f"dp {n}" for n in ["Logit", "NL (air travel)", PREF]]
                    + ["dprice_rivals", "CS loss ($m/yr)"]].rename(columns={"dp Logit": "logit (%)", "dp NL (air travel)": "NL (%)",
                                                                             f"dp {PREF}": "RC (%)", "dprice_rivals": "rivals, RC (%)"})
save_table(harm_tab, "b6nk_top_routes", float_format="%.2f",
           note=f"Twenty overlap routes with the largest consumer-surplus loss ({PREF}, {LATEST}). Fare changes in percent.")
harm_tab

In [ ]:
top_pct = rs[rs.passengers > 5000].sort_values(f"dp {PREF}", ascending=False).head(15).set_index("route")[
    [f"dp {n}" for n in results] + ["party_share", "parties_nonstop"]]
save_table(top_pct, "b6nk_top_pct_routes", float_format="%.2f",
           note="Fifteen overlap routes (at least 5,000 passengers per quarter) with the largest predicted B6/NK fare increase, by model.")
top_pct

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
vals = rs[[f"dp {n}" for n in ["Logit", "NL (air travel)", PREF]]].values
bins = np.linspace(min(-0.5, np.nanmin(vals)), max(10, np.nanpercentile(vals, 99)), 40)
for i, name in enumerate(["Logit", "NL (air travel)", PREF]):
    axes[0].hist(rs[f"dp {name}"], bins=bins, alpha=0.55, label=name, color=PALETTE[i])
axes[0].set_xlabel("Predicted B6/NK fare change (%)"); axes[0].set_ylabel("Overlap routes"); axes[0].legend()
h = harm.iloc[::-1].tail(15)
axes[1].barh(h.route, -4 * h.dcs_usd / 1e6, color=PALETTE[1])
axes[1].set_xlabel(f"Consumer-surplus loss, $m per year ({PREF})")
fig.tight_layout(); savefig(fig, "b6nk_fare_changes_and_harm")

## Efficiencies

**Compensating marginal-cost reductions (CMCR).** For each merging product, the cost reduction at which pre-merger
prices remain an equilibrium under joint ownership (Werden 1996): $c^* = p + (\Omega^{post}\circ J^{\top})^{-1}s$,
with $J_{jk}=\partial s_j/\partial p_k$. CMCR $=(c-c^*)/c$.

**Uniform efficiencies.** The post-merger equilibrium when B6 and NK marginal costs fall by 0–20%, and the reduction
that leaves consumer surplus unchanged.

In [ ]:
def cmcr(res, p, parties, markets):
    costs = res.compute_costs()[:, 0]
    pos = p.groupby("market_ids").indices
    out = []
    for m in markets:
        idx = pos[m]
        J = res.compute_demand_jacobians(market_id=m)
        s = p.shares.values[idx]; pr = p.prices.values[idx]
        firms = np.where(np.isin(p.carrier.values[idx], parties), parties[0], p.carrier.values[idx])
        O = (firms[:, None] == firms[None, :]).astype(float)
        cstar = pr + np.linalg.solve(O * J.T, s)
        for k, j in enumerate(idx):
            if p.carrier.values[j] in parties:
                out.append({"market_ids": m, "route": p.route.values[j], "carrier": p.carrier.values[j],
                            "nonstop": p.nonstop.values[j], "passengers": p.passengers.values[j],
                            "mc": costs[j] * 100, "cmcr_pct": 100 * (costs[j] - cstar[k]) / costs[j]})
    return pd.DataFrame(out)

MAIN = ["Logit", "NL (air travel)", PREF]
cm = {name: cmcr(results[name], sample, PARTIES, ov_latest) for name in MAIN}
grid = [0.0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20]
eff_rows = []
for name in MAIN:
    r = results[name]; costs = r.compute_costs()[:, 0]
    for e in grid:
        sim, cs = simulate_merger(r, sample, PARTIES, markets=ov_latest, efficiency=e, costs=costs)
        rr = route_summary(sim, cs, PARTIES)
        eff_rows.append({"model": name, "efficiency (%)": 100 * e, "B6/NK fare change (%)": party_avg(rr),
                         "rival fare change (%)": np.average(rr.dprice_rivals.fillna(0), weights=rr.passengers * (1 - rr.party_share)),
                         "CS change ($m/yr)": 4 * rr.dcs_usd.sum() / 1e6})
eff = pd.DataFrame(eff_rows)
breakeven = {}
for name, g in eff.groupby("model"):
    x, y = g["efficiency (%)"].values, g["CS change ($m/yr)"].values
    breakeven[name] = float(np.interp(0, y, x)) if (y.min() < 0 < y.max()) else (0.0 if y.min() >= 0 else np.nan)
cm_tab = pd.DataFrame({name: {"median CMCR (%)": d.cmcr_pct.median(),
                              "pax-weighted mean CMCR (%)": np.average(d.cmcr_pct, weights=d.passengers),
                              "B6 products, median (%)": d[d.carrier == "B6"].cmcr_pct.median(),
                              "NK products, median (%)": d[d.carrier == "NK"].cmcr_pct.median(),
                              "90th percentile (%)": d.cmcr_pct.quantile(0.9),
                              "CS break-even uniform reduction (%)": breakeven[name]} for name, d in cm.items()})
save_table(cm_tab, "b6nk_cmcr", float_format="%.2f",
           note="Compensating marginal-cost reductions for B6 and NK products on overlap routes, and the uniform percentage "
                "reduction in B6/NK marginal costs at which aggregate consumer surplus on overlap routes is unchanged.")
eff_w = eff.pivot(index="efficiency (%)", columns="model", values=["B6/NK fare change (%)", "CS change ($m/yr)"])
save_table(eff_w, "b6nk_efficiencies", float_format="%.2f",
           note="Post-merger equilibrium when B6 and NK marginal costs on overlap routes fall by the stated percentage.")
display(cm_tab); eff_w

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for i, (name, g) in enumerate(eff.groupby("model", sort=False)):
    axes[0].plot(g["efficiency (%)"], g["B6/NK fare change (%)"], marker="o", label=name, color=PALETTE[i])
    axes[1].plot(g["efficiency (%)"], g["CS change ($m/yr)"], marker="o", label=name, color=PALETTE[i])
for ax in axes:
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("Marginal-cost reduction for B6 and NK (%)")
axes[0].set_ylabel("B6/NK fare change (%)"); axes[1].set_ylabel("Consumer surplus change ($m per year)")
axes[0].legend(); fig.tight_layout(); savefig(fig, "b6nk_efficiencies")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
for i, (name, d) in enumerate(cm.items()):
    ax.hist(d.cmcr_pct.clip(-5, 60), bins=50, alpha=0.5, label=name, color=PALETTE[i])
ax.set_xlabel("Compensating marginal-cost reduction (%)"); ax.set_ylabel("B6/NK products"); ax.legend()
fig.tight_layout(); savefig(fig, "b6nk_cmcr")

## Seasonality: simulate in each of the four latest quarters

In [ ]:
season_rows = []
for name in ["NL (air travel)", PREF]:
    r = results[name]
    for (y, q) in SEASONS:
        mk = overlap_markets(sample, qlabel(y, q))
        sim, cs = simulate_merger(r, sample, PARTIES, markets=mk)
        rr = route_summary(sim, cs, PARTIES)
        season_rows.append({"model": name, "quarter": qlabel(y, q), "overlap routes": len(mk),
                            "B6/NK fare change (%)": party_avg(rr), "CS change ($m/qtr)": rr.dcs_usd.sum() / 1e6})
season = pd.DataFrame(season_rows).set_index(["model", "quarter"])
save_table(season, "b6nk_by_quarter", float_format="%.2f", note="Merger simulated separately in each base quarter.")
season

In [ ]:
if True:
    save_json({"summary": summary.to_dict(), "cmcr": cm_tab.to_dict(), "breakeven": breakeven,
               "efficiencies": eff.to_dict("records"), "by_quarter": season.reset_index().to_dict("records"),
               "n_overlap": len(ov_latest), "n_nonstop_overlap": int(ns_overlap.sum())}, "b6nk_results")